Cloud-only data dependency: this notebook reads the label Parquets written by `notebooks/prostate_subtype/nepc.ipynb`, `notebooks/prostate_subtype/avpc.ipynb`, and `notebooks/prostate_subtype/avpc_adjacent.ipynb`. It makes no LLM calls. The logic lives in `tasks/primary_subtype.py` (tested in `tests/test_primary_subtype.py`); this notebook only sets paths, prints diagnostics and writes.

# Primary Subtype Labels (NEPC > AVPC > AVPC-adjacent > PRAD)

Merges the full `nepc_labels.parquet`, `avpc_labels.parquet`, and `avpc_adjacent_labels.parquet`
tables (one row per patient, every column from each) and adds three columns.

`primary_label` is assigned in precedence order:

1. `nepc` — `nepc_status == "nepc"` (broad: any gated NE evidence, including t3 and IHC-only).
2. `avpc` — `nepc_status == "non_nepc"` and `is_avpc` is true.
3. `avpc_adjacent` — not NEPC, `is_avpc` is false, and `is_avpc_adjacent` is true.
4. `prad` — `nepc_status == "non_nepc"`, `is_avpc` is false, and `is_avpc_adjacent` is false.
   (This value was called `conventional` before AVPC-adjacent was added.)

`primary_label_definitive` is the same cascade keyed on `nepc_t1` (pathology-defined
definitive small-cell / large-cell / high-grade NE carcinoma) instead of the broad `nepc_status`.
Patients with only t2/t3/IHC NE evidence fall through to the AVPC / adjacent / prad branches.

`small_cell_discordant` flags patients whose AVPC C1 (or adjacent C1) small-cell finding is not
reflected by `nepc_status == "nepc"`. It changes no label; review those patients by hand.

Each label is null when a higher-precedence label is unresolved: that extraction failed, never
ran, or only partly ran for the patient (a partial review can establish a positive, never a
negative), or the patient is missing from one of the files.

Each input's `<prefix>_labels.meta.parquet` must match the gate and criteria versions in the
current code. Stale labels raise; re-run that pipeline with `REBUILD_LABELS_ONLY` (no LLM calls),
then re-run this notebook. The output gets `primary_subtype_labels.meta.parquet` recording each
input's path, sha256 and label metadata.

The two column names both NEPC and AVPC write get a pipeline prefix: `nepc_review_status` /
`avpc_review_status` and `nepc_gate_rejections` / `avpc_gate_rejections`. AVPC-adjacent columns
are already `avpc_adj_`-prefixed, except `is_avpc_adjacent` and `avpc_adjacent_date`. AVPC criteria
use the paper's numbering (C2 = exclusively visceral, C3 = lytic).

If `notebooks/prostate_subtype/arlpc.ipynb` has run, its `arlpc_`-prefixed AR-low / AR-indifferent columns are
left-joined on `DFCI_MRN` as annotation only: no label changes, and ARLPC columns are null for
patients absent from the ARLPC run.

In [ ]:
from pathlib import Path
import os
import sys

import polars as pl
from IPython.display import display


def find_repo_root(start):
    """Find the repo root (contains pyproject.toml) from the notebook directory."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the repo root (pyproject.toml)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tasks import primary_subtype

DEFAULT_DATA_ROOT = Path(
    os.environ.get("LLM_ANNOTATIONS_DATA_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/")
)
pl.Config.set_fmt_str_lengths(120)

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# Inputs: the OUTPUT_DIRs of notebooks/prostate_subtype/nepc.ipynb, avpc.ipynb, and avpc_adjacent.ipynb.
NEPC_LABELS_PATH = DEFAULT_DATA_ROOT / "LLM_nepc" / "nepc_labels.parquet"
AVPC_LABELS_PATH = DEFAULT_DATA_ROOT / "LLM_avpc" / "avpc_labels.parquet"
AVPC_ADJACENT_LABELS_PATH = DEFAULT_DATA_ROOT / "LLM_avpc_adjacent" / "avpc_adjacent_labels.parquet"
# Optional: AR-low / AR-indifferent annotation from notebooks/prostate_subtype/arlpc.ipynb (skipped if absent).
ARLPC_LABELS_PATH = DEFAULT_DATA_ROOT / "LLM_arlpc" / "arlpc_labels.parquet"

OUTPUT_DIR = DEFAULT_DATA_ROOT / "LLM_primary_subtype"
OUTPUT_PATH = OUTPUT_DIR / "primary_subtype_labels.parquet"

# Writing only replaces the merged file; no pipeline's outputs are touched.
WRITE_LABELS = True

for path in (NEPC_LABELS_PATH, AVPC_LABELS_PATH, AVPC_ADJACENT_LABELS_PATH, ARLPC_LABELS_PATH):
    print(f"{Path(path).exists()}: {path}")

In [ ]:
PATHS = {
    "nepc": NEPC_LABELS_PATH,
    "avpc": AVPC_LABELS_PATH,
    "avpc_adjacent": AVPC_ADJACENT_LABELS_PATH,
    "arlpc": ARLPC_LABELS_PATH,  # optional; skipped if absent
}
# Raises if a required file is missing or any input's label metadata is stale.
merged, inputs = primary_subtype.build(PATHS)

for name, info in inputs.items():
    print(f"{name}: {info['path']} (sha256 {info['sha256'][:12]})")
cohorts = {name: set(pl.read_parquet(PATHS[name])["DFCI_MRN"]) for name in primary_subtype.REQUIRED}
everyone = set.union(*cohorts.values())
print(f"In all three: {len(set.intersection(*cohorts.values()))} of {len(everyone)} patients")
for name, mrns in cohorts.items():
    print(f"  missing from {name}: {len(everyone - mrns)}")
if len({frozenset(m) for m in cohorts.values()}) > 1:
    print("WARNING: the runs used different cohorts. Patients missing from a file get a null "
          "primary_label unless a higher-precedence label is positive.")
if "arlpc" in inputs:
    print(f"ARLPC joined; {merged['arlpc_review_status'].null_count()} merged patients absent")
else:
    print(f"No ARLPC labels at {ARLPC_LABELS_PATH}; skipping that join")
print(f"Merged: {merged.height} patients, {merged.width} columns")

In [ ]:
print(merged["primary_label"].value_counts().sort("count", descending=True))
print(merged["primary_label_definitive"].value_counts().sort("count", descending=True))

# Broad vs definitive NEPC: who moves when only t1 counts as NEPC.
print(merged.group_by("primary_label", "primary_label_definitive").len()
      .sort("primary_label", "primary_label_definitive", nulls_last=True))

# Small-cell C1 that the NEPC pipeline did not label nepc: review by hand.
discordant = primary_subtype.small_cell_discordance(merged)
print(f"small_cell_discordant: {discordant.height} patients")
display(discordant)

In [ ]:
# NEPC x AVPC overlap: NEPC patients who also meet AVPC are labelled nepc.
print(merged.group_by("nepc_status", "is_avpc").len().sort("nepc_status", "is_avpc", nulls_last=True))

# AVPC x AVPC-adjacent overlap: strict AVPC patients are usually adjacent too (labelled avpc).
print(merged.group_by("is_avpc", "is_avpc_adjacent").len()
      .sort("is_avpc", "is_avpc_adjacent", nulls_last=True))

# Which adjacent criteria drove the avpc_adjacent label.
print(merged.filter(pl.col("primary_label") == "avpc_adjacent")
      .explode("avpc_adj_criteria").group_by("avpc_adj_criteria").len()
      .sort("len", descending=True))

# NEPC tier across primary labels.
print(merged.group_by("primary_label", "nepc_tier").len()
      .sort("primary_label", "nepc_tier", nulls_last=True))

# Why primary_label is null: the review status of each pipeline for those patients.
print(merged.filter(pl.col("primary_label").is_null())
      .group_by("nepc_review_status", "avpc_review_status", "avpc_adj_review_status").len()
      .sort("len", descending=True))

# AR-low / AR-indifferent across primary labels (annotation; precedence unchanged).
if "arlpc_strict" in merged.columns:
    print(merged.group_by("primary_label", "arlpc_strict", "arlpc_broad").len()
          .sort("primary_label", "arlpc_strict", "arlpc_broad", nulls_last=True))

display(merged.head(20))

In [ ]:
if WRITE_LABELS:
    meta_path = primary_subtype.write_outputs(merged, inputs, OUTPUT_PATH)
    print(f"Wrote {merged.height} rows to {OUTPUT_PATH} (provenance: {meta_path})")
else:
    print("Skipping write (WRITE_LABELS = False)")